In [1]:
import numpy as np
import pandas as pd

# =========================
# Reading csv file
# =========================
def read_csv(file_path):
    data = pd.read_csv(file_path)

    X = data.iloc[:, :-1].values.astype(float)
    Y = data.iloc[:, -1].values.reshape(-1, 1)

    X_min = X.min(axis=0)
    X_max = X.max(axis=0)

    X = (X - X_min) / (X_max - X_min + 1e-8)

    return X, Y

#=========================
# Activation Functions
# =========================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(a):
    return a * (1 - a)

def relu(z):
    return z if z > 0 else 0.0

def relu_prime(a):
    return 1.0 if a > 0 else 0.0


ACTIVATIONS = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "relu": (relu, relu_prime),
    
}

# =========================
# Loss Functions
# =========================

def mse_loss(a, y):
    return np.mean((a - y) ** 2)

def mse_dL_da(a, y):
    return (2 * (a - y)) / y.shape[0]   

def bce_loss(a, y):

    eps = 1e-12

    # keep values inside (0,1)
    a = np.clip(a, eps, 1 - eps)

    loss = -np.mean(
        y * np.log(a) +
        (1 - y) * np.log(1 - a)
    )

    return loss


def bce_dL_da(a, y):

    eps = 1e-12

    a = np.clip(a, eps, 1 - eps)

    grad = ((a - y) / (a * (1 - a))) / y.shape[0]

    return grad

LOSSES = {
    "mse": (mse_loss, mse_dL_da),
    "bce": (bce_loss, bce_dL_da),
}

# =========================
# Accuracy Function
# =========================
def accuracy(y_true, y_pred):

    # Convert probability → class label
    y_pred_class = (y_pred >= 0.5).astype(int)

    acc = np.mean(y_true == y_pred_class)

    return acc

# =========================
# Model
# =========================

class SingleLayerPerceptron:

    def __init__(self, n_inputs, activation="sigmoid",
                 loss="mse", learning_rate=0.1, seed=None):

        if seed is not None:
            np.random.seed(seed)

        self.learning_rate = learning_rate

        self.activation, self.activation_prime = ACTIVATIONS[activation]
        self.loss, self.dL_da = LOSSES[loss]

        self.weights = np.random.randn(n_inputs, 1) * 0.01
        self.bias = 0.0

    # -------------------------
    # Forward 
    # -------------------------
    def forward(self, X):

        Z = np.dot(X, self.weights) + self.bias
        A = self.activation(Z)

        return Z, A

    # -------------------------
    # Train 
    # -------------------------
    def train(self, X, Y):

        Z, A = self.forward(X)

        loss = self.loss(A, Y)

        # gradient chain rule
        dL_da = self.dL_da(A, Y)              
        da_dz= self.activation_prime(A)

        delta = dL_da * da_dz                
        m = X.shape[0]

        dL_dW = np.dot(X.T, delta)              
        dL_db = np.sum(delta) / m                

        # update
        self.weights -= self.learning_rate * dL_dW
        self.bias -= self.learning_rate * dL_db

        return loss, A

    # -------------------------
    # Prediction
    # -------------------------
    def predict(self, X):

        _, A = self.forward(X)
        return A
    
if __name__ == "__main__":

    X, Y = read_csv("heart.csv")

    # -------------------------
    # Train Test Split (80/20)
    # -------------------------

    np.random.seed(42)

    indices = np.random.permutation(len(X))

    X = X[indices]
    Y = Y[indices]

    split = int(0.8 * len(X))

    X_train = X[:split]
    Y_train = Y[:split]

    X_test = X[split:]
    Y_test = Y[split:]

    # -------------------------
    # Model
    # -------------------------

    model = SingleLayerPerceptron(
        n_inputs=X.shape[1],
        loss="bce",
        seed=42
    )

    # -------------------------
    # Training
    # -------------------------

    for epoch in range(5000):

        loss, pred = model.train(
            X_train,
            Y_train
        )

        if epoch % 200 == 0:

            train_acc = accuracy(
                Y_train,
                pred
            )

            print(
                f"Epoch {epoch} | "
                f"Loss: {loss:.4f} | "
                f"Train Accuracy: {train_acc:.4f}"
            )

    # -------------------------
    # Testing
    # -------------------------

    final_pred = model.predict(X_test)

    test_acc = accuracy(
        Y_test,
        final_pred
    )

    # print("\nFinal Prediction:")
    # print(final_pred)

    print(
        "\nTest Accuracy:",
        test_acc
    )
    

ModuleNotFoundError: No module named 'pandas'